# HPD 2 — Entregable: Agente analítico con Tool Calling

<figure>
<a
href="https://colab.research.google.com/github/Adamychen/m10_quarto/blob/main/notebooks/evaluables/hpd2-entregable.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

> **Peso en la nota:** 7.5 %
>
> **Plazo:** 7 días tras la sesión presencial.
>
> **Prerrequisito:** [HPD 2 — Agente analítico con Tool
> Calling](../bloque2/hpd2-agente-tool-calling.qmd).
>
> **Entrega:** Notebook `.ipynb` ejecutado con todas las celdas
> completas. Cada ejercicio especifica qué variable debe contener el
> resultado para la corrección automática.

> **Cómo se corrige**
>
> Cada ejercicio pide que asignes el resultado a una variable con un
> nombre concreto (`eval1_herramientas`, `eval2_analisis`,
> `eval3_pregunta`, `eval4_log`). El script de corrección ejecutará tu
> notebook e inspeccionará esas variables.

## Setup

In [1]:
!pip install -q langchain==1.3.2 langchain-openai==1.2.2 langchain-community==0.3.31 langchainhub "pandas<3" matplotlib seaborn python-dotenv==1.2.2 openai==2.38.0 scikit-learn langchain-core==1.4.0 langgraph==1.2.2

In [2]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import io
import base64

"""
Setup compartido para la API key de Llamus en todos los notebooks.
Inyectado en cada .qmd/.ipynb vía scripts/sync_setup_keys.py (pre-render).

Orden de resolución:
  1. .env local (python-dotenv)
  2. os.environ ya configurado externamente
  3. Google Colab userdata: secret "LLAMUS_API_KEY" (preferente) o "LLM_API_KEY"

Las celdas posteriores pueden leer la clave con os.environ.get("LLM_API_KEY").
"""

import os

def _load_llm_api_key():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

    if not os.getenv("LLM_API_KEY"):
        try:
            from google.colab import userdata
            for name in ("LLAMUS_API_KEY", "LLM_API_KEY"):
                try:
                    value = userdata.get(name)
                    if value:
                        os.environ["LLM_API_KEY"] = value
                        break
                except Exception:
                    continue
        except ImportError:
            pass

_load_llm_api_key()

if os.getenv("LLM_API_KEY"):
    print("✓ LLM_API_KEY cargada.")
else:
    print("⚠️  LLM_API_KEY no encontrada.")
    print("    • Local: crea un archivo .env con LLM_API_KEY=tu_clave")
    print("    • Colab: añade un Secret 🔑 con nombre 'LLAMUS_API_KEY'")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 100
print("Librerías cargadas correctamente.")

✓ LLM_API_KEY cargada.
Librerías cargadas correctamente.

In [3]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder, PromptTemplate

LLM_KEY = os.getenv("LLM_API_KEY")
LLM_URL = os.getenv("LLM_URL", "https://llamus.cs.us.es/api/v1")

if not LLM_KEY:
    print("⚠️  Crea un archivo .env con LLM_API_KEY=tu_clave para activar el agente.")
else:
    llm = ChatOpenAI(model=os.getenv("LLM_MODEL", "gemma4:26b"), base_url=LLM_URL, api_key=LLM_KEY, temperature=0, max_tokens=2048)
    print(f"✅ LLM configurado ({os.getenv('LLM_MODEL', 'gemma4:26b')}).")

✅ LLM configurado (gemma4:26b).

## Ejercicios — contexto de riesgo de crédito

**Contexto:** Trabajas como Data Scientist en el área de riesgos de un
banco (o de una consultora financiera). El banco tiene una cartera de
solicitudes de crédito al consumo y quiere saber qué perfiles tienen más
riesgo de impago. Tu cliente te pide un agente que consulte esa cartera
y responda preguntas sobre tasas de impago, importes solicitados y
perfiles de riesgo, usando herramientas especializadas en lugar de
inventar cifras.

### Dataset real — German Credit

> **Procedencia y licencia**
>
> El dataset es **Statlog (German Credit Data)**, de la UCI Machine
> Learning Repository (autor: Dr. Hans Hofmann, 1994). Se usa aquí a
> través de OpenML (`credit-g`, versión 1). Licencia: CC BY 4.0.
> Contiene 1000 solicitudes de crédito con 20 variables y la etiqueta
> `class` (`good` = buen pagador, `bad` = impago).
>
> **Aviso:** la celda siguiente usa `fetch_openml`, que **necesita
> conexión a internet** la primera vez que se ejecuta (después queda en
> caché local).

In [4]:
from sklearn.datasets import fetch_openml

try:
    german = fetch_openml("credit-g", version=1, as_frame=True, parser="auto")
except Exception as e:
    raise RuntimeError(
        "No se pudo descargar German Credit desde OpenML. "
        "Comprueba tu conexión a internet y vuelve a ejecutar la celda "
        "(fetch_openml necesita red la primera vez; después usa la caché local)."
    ) from e

df_a = german.frame.copy()
# Variable derivada: 1 = impago (class == 'bad'), 0 = buen pagador (class == 'good')
df_a["impago"] = (df_a["class"] == "bad").astype(int)

print(f"Dataset German Credit: {df_a.shape[0]} filas × {df_a.shape[1]} columnas")
print(f"Tasa de impago global: {df_a['impago'].mean():.1%}")
print(f"Propósitos de crédito: {df_a['purpose'].astype(str).unique().tolist()}")
display(df_a.head())

Dataset German Credit: 1000 filas × 22 columnas
Tasa de impago global: 30.0%
Propósitos de crédito: ['radio/tv', 'education', 'furniture/equipment', 'new car', 'used car', 'business', 'domestic appliance', 'repairs', 'other', 'retraining']

5 rows × 22 columns

#### Ejercicio 1 — Herramientas del agente (2 ptos)

Define al menos 2 herramientas (`@tool`) para el análisis de riesgo de
crédito. Deben incluir docstrings completos (args, returns, ejemplo) e
indicar que **devuelven como máximo 20 filas** para no saturar al LLM.
Posibilidades:

- `consultar_solicitudes(columna, valor, max_filas)` → solicitudes que
  cumplen un filtro (máx. 20 filas)
- `analizar_riesgo(proposito)` → número de solicitudes, tasa de impago,
  importe medio y duración media para un propósito, frente a la cartera
  total
- `comparar_grupos(columna)` → tabla de tasa de impago e importe medio
  por categoría de una variable (máx. 20 grupos)
- `visualizar_distribucion(columna)` → gráfico de la tasa de impago por
  categoría (si lo usas, devuelve al LLM solo una confirmación de texto,
  no la imagen)

Pista: para responder la pregunta del ejercicio 3 sin consultar cada
propósito por separado, `comparar_grupos` es la herramienta más útil.

Asigna tu lista de herramientas a `eval1_herramientas`.

In [5]:
# ─── COMPLETAR: define tus herramientas aquí ───

# Tool 1:
@tool
def consultar_solicitudes(columna: str, valor: str, max_filas: int = 10) -> str:
    """
    Consulta las solicitudes de crédito cuyo valor en 'columna' coincide con 'valor'.
    El DataFrame global 'df_a' contiene 1000 solicitudes del dataset German Credit.
    Columnas clave: purpose, credit_amount, duration, age, checking_status, savings_status,
    housing, job, class (good/bad) e impago (1 = impago, 0 = buen pagador).
    Devuelve como máximo 20 filas por respuesta (max_filas se limita a 20).

    Args:
        columna: Nombre de la columna por la que filtrar (p. ej. purpose, housing, job).
        valor: Valor exacto a buscar (p. ej. used car, rent, skilled).
        max_filas: Número máximo de filas a devolver, entre 1 y 20 (por defecto 10).

    Returns:
        String con el número total de solicitudes encontradas y las primeras filas (máx. 20).

    Ejemplo: consultar_solicitudes("purpose", "used car", 5)
    """
    # TODO: implementar
    resultado = f"Función pendiente de implementar para {columna}={valor}"
    return resultado

# Tool 2:
@tool
def analizar_riesgo(proposito: str) -> str:
    """
    Calcula el perfil de riesgo de un propósito de crédito: número de solicitudes,
    tasa de impago (%), importe medio solicitado, duración media (meses) y edad media,
    comparados con los valores de la cartera total.
    Devuelve un resumen corto (pocas líneas, no filas de datos).

    Args:
        proposito: Propósito del crédito tal como aparece en el dataset. Valores válidos:
            'new car', 'used car', 'furniture/equipment', 'radio/tv', 'domestic appliance',
            'repairs', 'education', 'retraining', 'business', 'other'.

    Returns:
        String con las métricas del propósito y de la cartera total.

    Ejemplo: analizar_riesgo("used car")
    """
    # TODO: implementar
    resultado = f"Función pendiente de implementar para {proposito}"
    return resultado

eval1_herramientas = [consultar_solicitudes, analizar_riesgo]
print("✅ eval1_herramientas asignada con 2 tools.")
print(f"   Tools: {[t.name for t in eval1_herramientas]}")

✅ eval1_herramientas asignada con 2 tools.
   Tools: ['consultar_solicitudes', 'analizar_riesgo']

#### Ejercicio 2 — Crear agente y ejecutar análisis (2.5 ptos)

Crea un agente ReAct con tus herramientas, configúralo con un system
prompt adecuado para el análisis de riesgo de crédito y ejecuta un
análisis concreto: *“Analiza las solicitudes de crédito para coche usado
(`purpose = 'used car'`): calcula cuántas hay, su tasa de impago y su
importe medio solicitado, y compara la tasa de impago con la de la
cartera completa.”*

Asigna el resultado a `eval2_analisis`.

In [6]:
# ─── COMPLETAR: crea el system prompt y el agente ───

system_prompt_a = """Eres un analista de riesgo de crédito experto. Usas herramientas para consultar
una cartera de solicitudes de crédito (dataset German Credit) y responder preguntas sobre tasas de
impago, importes solicitados y perfiles de riesgo. El impago es la clase 'bad' del dataset.
Siempre que necesites datos numéricos, usa tus herramientas en lugar de inventar valores.
Responde en español, de forma concisa y profesional."""

# TODO: crear el agente y ejecutar el análisis
#   agente = create_agent(model=llm, tools=eval1_herramientas, system_prompt=system_prompt_a)
#   result = agente.invoke({"messages": [{"role": "user", "content": pregunta}]})
#   eval2_analisis = result["messages"][-1].content

eval2_analisis = """Resultado del análisis de riesgo del propósito 'used car' pendiente de implementar."""
print("⚠️  Completa la implementación del agente y ejecuta el análisis.")

⚠️  Completa la implementación del agente y ejecuta el análisis.

#### Ejercicio 3 — Pregunta compleja multi-paso (1.5 ptos)

Formula una pregunta compleja que requiera múltiples pasos y
herramientas para responderse. Por ejemplo: *“¿Qué propósito de crédito
tiene la tasa de impago más alta entre los que tienen al menos 30
solicitudes? ¿Cuál es su importe medio solicitado y su duración media en
meses?”*.

Asigna la pregunta y la respuesta del agente a `eval3_pregunta` (como
diccionario con `pregunta` y `respuesta`).

In [7]:
pregunta_a = "¿Qué propósito de crédito tiene la tasa de impago más alta entre los que tienen al menos 30 solicitudes? ¿Cuál es su importe medio solicitado y su duración media en meses?"

# TODO: ejecutar la pregunta con el agente
#   result = agente.invoke({"messages": [{"role": "user", "content": pregunta_a}]})
#   respuesta_a = result["messages"][-1].content

eval3_pregunta = {
    "pregunta": pregunta_a,
    "respuesta": "Pendiente de implementar — ejecutar con el agente.",
}
print(f"✅ eval3_pregunta definida.")
print(f"   Pregunta: {eval3_pregunta['pregunta']}")

✅ eval3_pregunta definida.
   Pregunta: ¿Qué propósito de crédito tiene la tasa de impago más alta entre los que tienen al menos 30 solicitudes? ¿Cuál es su importe medio solicitado y su duración media en meses?

#### Ejercicio 4 — Log del razonamiento (1.5 ptos)

Ejecuta el agente con la pregunta del ejercicio anterior y captura el
log de razonamiento (la secuencia de pensamientos, llamadas a tools y
observaciones). Asigna los pasos intermedios a `eval4_log`.

In [8]:
# TODO: ejecutar la pregunta y capturar el log de razonamiento.
#   Con create_agent, el razonamiento está en result["messages"]:
#   - AIMessage con tool_calls: qué tool llamó el agente y con qué argumentos
#   - ToolMessage: la observación devuelta por la tool
#   Recorre result["messages"] y guarda cada paso (tool_call → observación) en eval4_log.

eval4_log = "Log del razonamiento pendiente de implementar."
print("⚠️  Completa la captura del log de razonamiento del agente.")

⚠️  Completa la captura del log de razonamiento del agente.

## Resumen de entregables

In [9]:
print(f"\nVariables de corrección:")
for var in ["eval1_herramientas", "eval2_analisis", "eval3_pregunta", "eval4_log"]:
    val = globals().get(var, None)
    status = "✅ definida" if val is not None else "⚠️ no definida"
    print(f"  {var}: {status}")


Variables de corrección:
  eval1_herramientas: ✅ definida
  eval2_analisis: ✅ definida
  eval3_pregunta: ✅ definida
  eval4_log: ✅ definida

# References